# DAE/N2N/REO/denoise(empiar10311_stack_crop)

In [ ]:
%run ../../params.ipynb

In [ ]:
from pathlib import Path
from collections import namedtuple
import json

In [ ]:
Args = namedtuple("args", ["original", "even", "odd", "even_reg", "odd_reg", "denoised"])
args = Args("/home/vruiz/volumes/vols/empiar10311_stack_crop.mrc",
            "../even_odd/even.mrc", "../even_odd/odd.mrc",
            "even_registered.mrc", "odd_registered.mrc", 
            "denoised")

In [ ]:
if Path(args.denoised).exists():
    raise Exception(f"{args.denoised} already exists ... exiting")

## Configure cryoCARE

In [ ]:
_ = {
    "even": [args.even, args.even_reg],
    "odd": [args.odd_reg, args.odd],
    "mask": [""],
    "patch_shape": patch_shape,
    "num_slices": num_slices,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": n_normalization_samples,
    "path": "./data",
    "overwrite": "True"
}

with open("train_data_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_data_config.json

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar10311
#source ~/envs/cryoCARE/bin/activate
cryoCARE_extract_train_data.py --conf train_data_config.json

## Train

In [ ]:
_ = {
    "train_data": "./data",
    "epochs": epochs,
    "steps_per_epoch": steps_per_epoch,
    "batch_size": batch_size,
    "unet_kern_size": 3,
    "unet_n_depth": 3,
    "unet_n_first": 16,
    "learning_rate": 0.0004,
    "model_name": "model",
    "path": "./",
    "gpu_id": [0]
}

with open("train_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_config.json

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar10311
#source ~/envs/cryoCARE/bin/activate
cryoCARE_train.py --conf train_config.json

## Infer

In [ ]:
_ = {
    "path": "./model.tar.gz",
    "even": [args.original], 
    "odd": [args.original],
    "n_tiles": [4,4,8],
    "output": args.denoised,
    "overwrite": "True",
    "gpu_id": [0]
}

with open("predict_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat predict_config.json

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar10311
#pwd
#source ~/envs/cryoCARE/bin/activate
cryoCARE_predict.py --conf predict_config.json || true

In [ ]:
!pwd

In [ ]:
!ls -l denoised